# Notebook 03 — Topics and Concern Drivers

**Project:** Detecting Service Gaps and Bias Signals in Hospital Reviews  

---

## Goals
1. Apply LDA topic modelling to surface latent complaint themes in negative reviews
2. Identify which concern signal categories are the strongest **drivers of negative sentiment**
3. Visualise topic distribution and concern co-occurrence

## 0 — Setup

In [ ]:
import sys
import os
from pathlib import Path

ROOT = Path(os.getcwd()).parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import re
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

matplotlib.rcParams['figure.dpi'] = 120

FIGURES_DIR = ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

## 1 — Load Flagged Data

In [ ]:
flagged_path = ROOT / 'data' / 'processed' / 'reviews_flagged.csv'

if flagged_path.exists():
    df = pd.read_csv(flagged_path)
    print(f'Loaded flagged data: {df.shape}')
else:
    # Fallback: load raw and compute flags inline
    df = pd.read_csv(ROOT / 'data' / 'hospital.csv')
    df.columns = df.columns.str.strip()
    df = df.rename(columns={
        'Feedback':        'review_text',
        'Sentiment Label': 'sentiment_label',
        'Ratings':         'rating',
    })
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    df = df.drop_duplicates().dropna(subset=['review_text'])
    df['sentiment_label'] = pd.to_numeric(df['sentiment_label'], errors='coerce').fillna(0).astype(int)
    df['sentiment'] = df['sentiment_label'].map({1: 'positive', 0: 'negative'})
    df = df.reset_index(drop=True)

    CONCERN_LEXICON = {
        'mistreatment': ['rude','rudely','ignored','dismissive','disrespectful','insulted','arrogant','negligent','unprofessional'],
        'access_barriers': ['wheelchair','ramp','accessible','lift','elevator','stairs','disabled','handicap','parking'],
        'delays_wait': ['waiting','wait','waited','hours','queue','delayed','delay','slow','late','forever','long time'],
        'safety_hygiene': ['dirty','unclean','unhygienic','hygiene','infection','infested','smell','smelly','contaminated'],
    }
    for cat, kws in CONCERN_LEXICON.items():
        df[f'flag_{cat}'] = df['review_text'].apply(
            lambda x: int(any(kw in str(x).lower() for kw in kws)))
    flag_cols = [f'flag_{c}' for c in CONCERN_LEXICON]
    df['flag_any_concern'] = (df[flag_cols].sum(axis=1) > 0).astype(int)

flag_cols = [c for c in df.columns if c.startswith('flag_') and c != 'flag_any_concern']
print('Flag columns:', flag_cols)
df.head(3)

## 2 — Concern Drivers of Negative Sentiment

Which concern categories are most predictive/associated with negative sentiment labels?

In [ ]:
# Compute flag rate within negative vs positive reviews
rate_neg = df[df['sentiment_label']==0][flag_cols].mean() * 100
rate_pos = df[df['sentiment_label']==1][flag_cols].mean() * 100
lift = (rate_neg - rate_pos).sort_values(ascending=True)

categories = [c.replace('flag_','').replace('_',' ') for c in lift.index]

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in lift.values]
bars = ax.barh(categories, lift.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Concern Rate Difference: Negative − Positive (%)')
ax.set_title('Concern Category Lift in Negative Reviews\n(positive = stronger driver of negative sentiment)', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for bar, val in zip(bars, lift.values):
    x_pos = val + 0.3 if val >= 0 else val - 0.3
    ax.text(x_pos, bar.get_y() + bar.get_height()/2, f'{val:+.1f}%',
            va='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'concern_lift_by_category.png', bbox_inches='tight')
plt.show()
print('Saved → reports/figures/concern_lift_by_category.png')

## 3 — Concern Co-occurrence Heatmap

Do certain types of concerns tend to appear together?

In [ ]:
co_matrix = df[flag_cols].T.dot(df[flag_cols])
labels = [c.replace('flag_','').replace('_','\n') for c in flag_cols]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(co_matrix.values, cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='Co-occurrence count')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_title('Concern Category Co-occurrence', fontsize=12, fontweight='bold')
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, str(co_matrix.values[i, j]), ha='center', va='center',
                fontsize=9, color='black')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'concern_cooccurrence.png', bbox_inches='tight')
plt.show()
print('Saved → reports/figures/concern_cooccurrence.png')

## 4 — LDA Topic Modelling on Negative Reviews

LDA surfaces complaint themes that go **beyond our predefined lexicon**.  
We run it on negative reviews only, where the actionable intelligence lies.

In [ ]:
STOPWORDS = {
    'the','a','an','and','or','but','in','on','at','to','for','of','with','is','are',
    'was','were','be','been','have','had','has','i','we','they','it','this','that',
    'my','our','their','very','not','no','so','as','by','do','did','does','its',
    'from','up','about','into','than','also','just','more','all','would','will',
    'can','could','should','here','there','even','good','great','hospital'
}

neg_reviews = df[df['sentiment_label'] == 0]['review_text'].dropna()

def clean_for_lda(text):
    tokens = re.findall(r'[a-z]+', str(text).lower())
    return ' '.join(t for t in tokens if t not in STOPWORDS and len(t) > 2)

neg_clean = neg_reviews.apply(clean_for_lda)
neg_clean = neg_clean[neg_clean.str.len() > 5]

print(f'Negative reviews for LDA: {len(neg_clean):,}')
print('Sample cleaned text:', neg_clean.iloc[0][:100])

In [ ]:
NUM_TOPICS = 6

vectorizer = CountVectorizer(max_features=2000, min_df=2)
dtm = vectorizer.fit_transform(neg_clean)

lda = LatentDirichletAllocation(
    n_components=NUM_TOPICS,
    random_state=42,
    max_iter=20,
    learning_method='online',
)
lda.fit(dtm)
print(f'LDA fitted with {NUM_TOPICS} topics on {dtm.shape[0]:,} documents and {dtm.shape[1]:,} features.')

In [ ]:
# Display top keywords per topic
vocab = vectorizer.get_feature_names_out()
N_WORDS = 10

topic_top_words = {}
print('=== LDA Topics (Top Keywords) ===\n')
for topic_idx, topic in enumerate(lda.components_):
    top_ids = topic.argsort()[:-N_WORDS - 1:-1]
    top_words = [vocab[i] for i in top_ids]
    topic_top_words[f'Topic {topic_idx+1}'] = top_words
    print(f'Topic {topic_idx+1}: {"  ".join(top_words)}')

In [ ]:
# Visualise LDA topics as horizontal bars
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

palette = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6']

for topic_idx, topic in enumerate(lda.components_):
    top_ids = topic.argsort()[:-N_WORDS - 1:-1]
    top_words = [vocab[i] for i in top_ids]
    top_scores = topic[top_ids]

    ax = axes[topic_idx]
    ax.barh(list(reversed(top_words)), list(reversed(top_scores)),
            color=palette[topic_idx], edgecolor='white')
    ax.set_title(f'Topic {topic_idx+1}', fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('LDA Topics — Negative Hospital Reviews', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'lda_topics.png', bbox_inches='tight')
plt.show()
print('Saved → reports/figures/lda_topics.png')

## 5 — Topic Distribution Across Negative Reviews

In [ ]:
# Assign dominant topic to each negative review
doc_topics = lda.transform(dtm)
dominant_topic = doc_topics.argmax(axis=1)
topic_dist = pd.Series(dominant_topic).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([f'Topic {i+1}' for i in topic_dist.index], topic_dist.values,
       color=palette[:len(topic_dist)], edgecolor='white')
ax.set_title('Distribution of Dominant Topics\n(Negative Reviews)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Reviews')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'topic_distribution.png', bbox_inches='tight')
plt.show()
print('Saved → reports/figures/topic_distribution.png')

---
## Summary

- Identified which concern categories are **strongest drivers** of negative sentiment (lift analysis)
- Showed **concern co-occurrence** patterns
- LDA surfaced **latent complaint themes** beyond the pre-defined lexicon

**Manually label the topics above** with descriptive names based on the top keywords (e.g., Topic 3 → "Discharge Process Issues").  

**Next:** `04_model_baselines.ipynb` — TF-IDF + Logistic Regression baseline